# Market vs Weather Baselines

This notebook inspects the `weather_nyc_smoke_v1_features` dataset and compares Kalshi market prices with simple weather-threshold baseline signals.

In [ ]:
from pathlib import Path
import polars as pl

DATASET = "weather_nyc_smoke_v1_features"
files = list(Path("../data/processed/datasets").joinpath(DATASET).rglob("*.parquet"))
df = (
    pl.concat([pl.read_parquet(f) for f in files], how="diagonal_relaxed")
    if files
    else pl.DataFrame()
)
df.head()

In [ ]:
summary = {
    "rows": len(df),
    "markets": df.select("market_ticker").n_unique()
    if "market_ticker" in df.columns and len(df)
    else 0,
    "missing_forecasts": df.filter(pl.col("forecast_temperature").is_null()).height
    if "forecast_temperature" in df.columns
    else None,
    "missing_labels": df.filter(pl.col("label").is_null()).height
    if "label" in df.columns
    else None,
}
summary

In [ ]:
if len(df) and {"forecast_event_indicator", "label"}.issubset(df.columns):
    resolved = df.filter(
        pl.col("label").is_not_null() & pl.col("forecast_event_indicator").is_not_null()
    )
    accuracy = (
        (resolved["forecast_event_indicator"] == resolved["label"]).mean()
        if len(resolved)
        else None
    )
    print("forecast_event_indicator_accuracy", accuracy)
else:
    print("Required columns not available yet.")

In [ ]:
if len(df) and {"forecast_minus_threshold", "market_mid", "label"}.issubset(df.columns):
    bucketed = df.with_columns(
        pl.when(pl.col("forecast_minus_threshold") <= -3)
        .then(pl.lit("forecast strongly below"))
        .when(pl.col("forecast_minus_threshold") < 3)
        .then(pl.lit("forecast near threshold"))
        .otherwise(pl.lit("forecast strongly above"))
        .alias("bucket")
    )
    bucketed.group_by("bucket").agg(
        pl.len().alias("n_obs"),
        pl.col("market_mid").mean().alias("avg_market_mid"),
        pl.col("label").mean().alias("empirical_yes_rate"),
        pl.col("market_spread").mean().alias("avg_spread"),
    )
else:
    pl.DataFrame()

In [ ]:
if len(df) and {"market_mid", "forecast_event_indicator"}.issubset(df.columns):
    df.with_columns(
        (pl.col("market_mid") / 100 - pl.col("forecast_event_indicator"))
        .abs()
        .alias("market_vs_forecast_gap")
    ).sort("market_vs_forecast_gap", descending=True).head(10)
else:
    pl.DataFrame()

## Initial Finding

Run this notebook after `scripts/run_weather_smoke_test.sh`. The first conclusion should compare the naive NWS threshold signal with Kalshi midpoint and identify whether the largest disagreements occur near or far from the threshold.